# schedule_failure_mode_notify_v1

目标：验证 dashboard refresh 的详细通知可以携带 top failure mode，让用户知道哪个模型在什么 regime/window 下最需要 review。

In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists() and (path / 'src').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.schedule import ModelDashboardPipelineConfig, run_model_dashboard_refresh_once

CARD = REPO_ROOT / 'research/schedule_failure_mode_notify_v1'
LOCKS = CARD / 'locks'
CARD.mkdir(parents=True, exist_ok=True)
{'card': str(CARD.relative_to(REPO_ROOT))}

{'card': 'research/schedule_failure_mode_notify_v1'}

## 构造 dashboard payload

matrix runner 只创建 card 目录；dashboard exporter 返回带 `failure_modes` 的 dashboard payload。

In [2]:
def matrix_runner(**kwargs):
    Path(kwargs['card_dir']).mkdir(parents=True, exist_ok=True)
    return {'status': 'feature_set_matrix_completed', 'variant_count': 2}

def dashboard_exporter(card_dir: Path, *, output_path: Path, top_n: int):
    payload = {
        'schema_version': 'dashboard.model_matrix.v1',
        'artifact_id': 'failure_mode_notify_card',
        'top_models': [{'model': 'ridge_mse', 'rank': 1}],
        'manifest_source': {
            'kind': 'analysis_manifest',
            'path': str(card_dir / 'analysis_manifest.json'),
            'schema_version': 'artifact.v1',
            'manifest_hash': 'c' * 64,
            'producer': 'src.analysis.exports',
        },
        'failure_modes': [{
            'model': 'gcn_mse',
            'segment': 'stress_window::stress',
            'failure_mode': 'return_stability_lag',
            'severity_score': 78.04619,
        }],
    }
    output_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    return payload


## 运行 schedule refresh

开启 `notify_details=True` 后，通知应包含 `top_failure=return_stability_lag@stress_window::stress`。

In [3]:
events = []
config = ModelDashboardPipelineConfig(
    name='dashboard_failure_notify',
    card_dir=CARD / 'matrix_card',
    dashboard_output_path=CARD / 'dashboard_export.json',
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse', 'gcn_mse'),
    notify_details=True,
)
result = run_model_dashboard_refresh_once(
    config,
    lock_dir=LOCKS,
    notify=events.append,
    matrix_runner=matrix_runner,
    dashboard_exporter=dashboard_exporter,
)
assert result.status == 'success'
assert result.payload['top_failure_mode'] == 'return_stability_lag'
assert result.payload['top_failure_segment'] == 'stress_window::stress'
assert 'top_failure=return_stability_lag@stress_window::stress' in events[0]
events

['dashboard_failure_notify:success | top_model=ridge_mse | dashboard_manifest=analysis_manifest | attempts=1 | export=/Users/rziiiii/dev/workspace/research/schedule_failure_mode_notify_v1/dashboard_export.json | top_failure=return_stability_lag@stress_window::stress']

In [4]:
summary = {
    'schema_version': 'research.schedule_failure_mode_notify_v1',
    'status': 'completed',
    'notification': events[0],
    'top_failure_mode': result.payload['top_failure_mode'],
    'top_failure_segment': result.payload['top_failure_segment'],
    'top_failure_model': result.payload['top_failure_model'],
    'top_failure_severity': result.payload['top_failure_severity'],
    'conclusion': 'schedule 详细通知已能携带 dashboard failure_modes 的最高优先级失败模式。',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
(CARD / 'notification_events.json').write_text(json.dumps({'events': events, 'payload': result.payload}, ensure_ascii=False, indent=2), encoding='utf-8')
summary

{'schema_version': 'research.schedule_failure_mode_notify_v1',
 'status': 'completed',
 'notification': 'dashboard_failure_notify:success | top_model=ridge_mse | dashboard_manifest=analysis_manifest | attempts=1 | export=/Users/rziiiii/dev/workspace/research/schedule_failure_mode_notify_v1/dashboard_export.json | top_failure=return_stability_lag@stress_window::stress',
 'top_failure_mode': 'return_stability_lag',
 'top_failure_segment': 'stress_window::stress',
 'top_failure_model': 'gcn_mse',
 'top_failure_severity': 78.04619,
 'conclusion': 'schedule 详细通知已能携带 dashboard failure_modes 的最高优先级失败模式。'}

## 中文结论

- 做了什么：把 dashboard `failure_modes` 的最高优先级失败模式接入 schedule 详细通知。
- 核心指标：通知包含 `top_failure=return_stability_lag@stress_window::stress`，payload 同时保留 failure mode、segment、model 和 severity。
- regime 差异：top failure 指向 stress window，帮助 review 优先看压力期模型稳定性和相对收益。
- 是否晋升：这是 schedule/dashboard 联动能力，可保留在 `src/schedule`；不是策略晋升。
- 风险：只有 dashboard payload 已提供 `failure_modes` 且 `notify_details=True` 时才追加该字段，默认通知兼容旧格式。
- 下一步：dashboard 页面可以把 failure modes 做成独立卡片，通知只保留最高优先级摘要。